# Module 2: Build The Metadata Graph

This notebook walks through the full Text2SQL pipeline:
1. Connect to BigQuery and Neo4j
2. Extract BigQuery metadata (ETL: extract → transform → load)
3. Generate vector embeddings for semantic search
4. Explore the resulting Neo4j graph

**Prerequisites:** Complete `workshop/setup/environment-setup.md` before running this notebook.

## Setup — Imports and Environment

**Sync project dependencies before running this notebook!**

Install libraries to the `uv` managed environment.
```bash
uv sync
```

In [14]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Verify key variables are set
required_vars = [
    'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE',
    'GCP_PROJECT_ID', 'BIGQUERY_DATASET_ID', 'OPENAI_API_KEY'
]

print("Environment variable check:")
for var in required_vars:
    val = os.getenv(var)
    status = '✅' if val and val not in ('...', 'neo4j-uri', 'neo4j-password') else '❌'

    print(f"  {status} {var}")

Environment variable check:
  ✅ NEO4J_URI
  ✅ NEO4J_USERNAME
  ✅ NEO4J_PASSWORD
  ✅ NEO4J_DATABASE
  ✅ GCP_PROJECT_ID
  ✅ BIGQUERY_DATASET_ID
  ✅ OPENAI_API_KEY


Acme Corp is the default demo dataset, but others may be provided in the environment variable `BIGQUERY_DATASET_ID`

In [2]:
DATASET_ID = os.getenv('BIGQUERY_DATASET_ID', 'acme_corp')

## Confirm Connections

In [15]:
from neo4j import GraphDatabase
from openai import OpenAI
from google.cloud import bigquery

# Initialize Neo4j driver
neo4j_driver = GraphDatabase.driver(
    uri=os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD')),
)
neo4j_database = os.getenv('NEO4J_DATABASE', 'neo4j')

# Verify Neo4j connection
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run('RETURN 1 AS ping')
    message = '✅' if result.single()['ping'] else '❌'
    print(f"{message} Neo4j driver connected")

# Initialize OpenAI client
embedding_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print("✅ OpenAI client initialized")

# Initialize BigQuery client
bigquery_client = bigquery.Client(project=os.getenv('GCP_PROJECT_ID'))
print(f"✅ BigQuery client connected to project {bigquery_client.project}")

✅ Neo4j driver connected
✅ OpenAI client initialized
✅ BigQuery client connected to project ai-field-alex-g


## Load the Sample Dataset into BigQuery

This step creates the `acme_corp` dataset with 33 tables from the ddl file found [here](../datasets/ddl/acme-dataset.sql).

The Acme Corp dataset is a comprehensive B2B SaaS/software company operational data model.

*Skip this cell if the dataset already exists in BigQuery.*

In [4]:
from pathlib import Path

The loading process many take a few minutes to complete.

In [5]:
with Path("../datasets/demo/ddl/acme-dataset.sql").open() as f:
    sql = f.read()

    job = bigquery_client.query(sql)
    job.result()

In [7]:
# Verify the tables exist in BigQuery
project_id = os.getenv('GCP_PROJECT_ID')

tables = list(bigquery_client.list_tables(f"{project_id}.{DATASET_ID}"))
print(f"Tables in {project_id}.{DATASET_ID}:")
for table in tables[:5]:
    print(f"  - {table.table_id}")
print(f"\nTotal tables: {len(tables)}")

Tables in ai-field-alex-g.acme_corp:
  - campaigns
  - compensation
  - customer_addresses
  - customer_contacts
  - customers

Total tables: 33


## Extract BigQuery Metadata

The `BigQuerySchemaConnector` reads `INFORMATION_SCHEMA` tables to extract:
- Table names and descriptions
- Column names, types, nullability
- Primary key and foreign key constraints

In [8]:
from neocarta.connectors.bigquery import BigQuerySchemaConnector

In [16]:
# Initialize the BigQuery workflow
bigquery_connector = BigQuerySchemaConnector(
    client=bigquery_client,
    project_id=os.getenv('GCP_PROJECT_ID'),
    dataset_id=DATASET_ID,
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database,
)

This step may take a few minutes

In [17]:
# Step 1: Extract metadata from BigQuery INFORMATION_SCHEMA
print("Extracting metadata from BigQuery...")
bigquery_connector.extract_metadata()
print("Extraction complete.")

Extracting metadata from BigQuery...
Extraction complete.


In [18]:
# Inspect what was extracted — results are cached on the extractor
print(f"\nExtracted raw metadata:")
print(f"  Tables: {len(bigquery_connector.extractor.table_info)}")
print(f"  Columns: {len(bigquery_connector.extractor.column_info)}")
print(f"  FK references: {len(bigquery_connector.extractor.column_references_info)}")


Extracted raw metadata:
  Tables: 33
  Columns: 330
  FK references: 87


## Transform Raw Metadata into Pydantic Models

The transformer converts raw BigQuery rows into typed Pydantic models:
- `Database`, `Schema`, `Table`, `Column`, `Value`
- Hierarchical relationships
- `References` for FK relationships

Pydantic validation catches data quality issues before anything hits Neo4j.

In [19]:
# Step 2: Transform raw data into Pydantic models
print("Transforming metadata into graph schema...")
bigquery_connector.transform_metadata()
print("\nTransformation complete.")

Transforming metadata into graph schema...

Transformation complete.


In [20]:
# The transformer caches results — inspect them
# Import the models to understand the structure
from neocarta.data_model.rdbms import Database, Schema, Table, Column

print("\nData model nodes:")
print(f"- Database fields: {list(Database.model_fields.keys())}")
print(f"- Schema fields  : {list(Schema.model_fields.keys())}")
print(f"- Table fields   : {list(Table.model_fields.keys())}")
print(f"- Column fields  : {list(Column.model_fields.keys())}")


Data model nodes:
- Database fields: ['id', 'name', 'platform', 'service', 'description', 'embedding']
- Schema fields  : ['id', 'name', 'description', 'embedding']
- Table fields   : ['id', 'name', 'description', 'embedding']
- Column fields  : ['id', 'name', 'description', 'embedding', 'type', 'nullable', 'is_primary_key', 'is_foreign_key']


## Load into Neo4j

The loader uses `MERGE` Cypher queries to load nodes and relationships into Neo4j.

`MERGE` is idempotent — safe to re-run without creating duplicates.

In [21]:
# Step 3: Load transformed data into Neo4j
print("Loading metadata into Neo4j...")
bigquery_connector.load_metadata()
print("\nLoad complete.")

Loading metadata into Neo4j...
{'_contains_updates': True, 'labels_added': 1, 'nodes_created': 1, 'properties_set': 3}
{'_contains_updates': True, 'labels_added': 1, 'nodes_created': 1, 'properties_set': 3}
{'_contains_updates': True, 'labels_added': 33, 'nodes_created': 33, 'properties_set': 99}
{'_contains_updates': True, 'labels_added': 327, 'nodes_created': 327, 'properties_set': 2289}
{'_contains_updates': True, 'labels_added': 2351, 'nodes_created': 2351, 'properties_set': 4702}
{'_contains_updates': True, 'relationships_created': 1}
{'_contains_updates': True, 'relationships_created': 33}
{'_contains_updates': True, 'relationships_created': 327}
{'_contains_updates': True, 'relationships_created': 54, 'properties_set': 54}
{'_contains_updates': True, 'relationships_created': 2351}

Load complete.


Update the Neocarta metadata node to track which version this graph was built with. This is automatic when using the `.run()` method, but since we're walking through the internal method calls, we have to run the upsert method manually.

In [23]:
print(bigquery_connector.loader.upsert_neocarta_graph_node().model_dump())

{'initial_version': '0.4.0', 'latest_version': '0.4.0', 'create_date': datetime.datetime(2026, 5, 20, 21, 5, 36, 112000, tzinfo=<UTC>), 'last_updated': datetime.datetime(2026, 5, 20, 21, 5, 53, 288000, tzinfo=<UTC>)}


Confirm node and relationship counts with the Neo4j driver.

In [24]:
# Verify the node counts in Neo4j
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count ASC"
    )
    print("\nNode counts in Neo4j:")
    for record in result:
        print(f"- {record['label']}: {record['count']}")

# Verify FK relationships
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run(
        "MATCH ()-[r:REFERENCES]->() RETURN count(r) AS fk_count"
    )
    fk_count = result.single()['fk_count']
    print(f"\nForeign key relationships: {fk_count}")


Node counts in Neo4j:
- Database: 1
- Schema: 1
- __neocarta_graph__: 1
- Table: 33
- Column: 327
- Value: 2351

Foreign key relationships: 54


## Generate Vector Embeddings

The `OpenAIEmbeddingsConnector` can generate embeddings for nodes containing `description` properties.

Default settings are:
- 768 dimensions
- `text-embedding-3-small` embedding model

Process:
1. Creates a vector index for each node label (if it doesn't exist)
2. Finds all nodes with a `description` field but no `embedding`
3. Calls OpenAI in batches of 100
4. Writes embeddings back to Neo4j

We will generate embeddings for `Schema`, `Table` and `Column` since we will use them as entry points into our semantic graph.

**Note: This will generate 361 embeddings for the `acme_corp` dataset! This will incur cost!**

In [25]:
from neocarta.enrichment.embeddings import OpenAIEmbeddingsConnector

node_labels = ['Schema', 'Table', 'Column']

openai_embedding_connector = OpenAIEmbeddingsConnector(
    client=embedding_client,
    embedding_model='text-embedding-3-small',
    dimensions=768,
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database,
)

print("Generating embeddings for node labels:", node_labels)

openai_embedding_connector.run(node_labels=node_labels)

print("Embeddings complete!")

Generating embeddings for node labels: ['Schema', 'Table', 'Column']
Processing Schema nodes...
--------------------------------
Processing batch 1 of 1  
Successful Embeddings : 1
{}
Processing Table nodes...
--------------------------------
Processing batch 1 of 1  
Successful Embeddings : 33
{}
Processing Column nodes...
--------------------------------
Processing batch 1 of 4  
Processing batch 2 of 4  
Processing batch 3 of 4  
Processing batch 4 of 4  
Successful Embeddings : 327
{}
Embeddings complete!


In [26]:
# Verify embedding coverage
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (n)
        WHERE n.embedding IS NOT NULL OR n.embedding IS NULL
        WITH labels(n)[0] AS label, n
        RETURN label,
               count(n) AS total,
               count(n.embedding) AS with_embedding
        ORDER BY label
    """)
    print("Embedding coverage:")
    for record in result:
        status = '✅' if record['total'] == record['with_embedding'] else '⚠️'
        print(f"  {status} {record['label']}: {record['with_embedding']}/{record['total']} nodes embedded")

Embedding coverage:
  ✅ Column: 327/327 nodes embedded
  ⚠️ Database: 0/1 nodes embedded
  ✅ Schema: 1/1 nodes embedded
  ✅ Table: 33/33 nodes embedded
  ⚠️ Value: 0/2351 nodes embedded
  ⚠️ __neocarta_graph__: 0/1 nodes embedded


## Explore the Graph

Use the Neo4j Python driver to run Cypher queries and inspect what was loaded.

Now run some exploratory Cypher queries.

In [51]:
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
        RETURN t.name AS table,
               collect({name: c.name, type: c.type, pk: c.is_primary_key, fk: c.is_foreign_key}) AS columns
        ORDER BY t.name
        LIMIT 3
    """)
    print("Schema Examples:")
    print("-" * 60)
    for record in result:
        print(f"\nTable: {record['table']}")
        for col in record['columns']:
            flags = []
            if col['pk']: flags.append('PK')
            if col['fk']: flags.append('FK')
            flag_str = f" [{', '.join(flags)}]" if flags else ""
            print(f"  - {col['name']} ({col['type']}){flag_str}")

Schema Examples:
------------------------------------------------------------

Table: campaigns
  - campaign_id (STRING) [PK]
  - name (STRING)
  - channel (STRING)
  - start_date (DATE)
  - end_date (DATE)
  - budget_usd (NUMERIC)
  - spend_usd (NUMERIC)
  - owner_employee_id (STRING)

Table: compensation
  - compensation_id (STRING) [PK]
  - employee_id (STRING)
  - effective_date (DATE)
  - base_salary (NUMERIC)
  - bonus_target_pct (NUMERIC)
  - equity_grant_usd (NUMERIC)
  - currency (STRING)
  - change_type (STRING)
  - approved_by (STRING)

Table: customer_addresses
  - address_id (STRING) [PK]
  - customer_id (STRING)
  - address_type (STRING)
  - line1 (STRING)
  - line2 (STRING)
  - city (STRING)
  - region (STRING)
  - postal_code (STRING)
  - country (STRING)
  - is_primary (BOOL)


In [49]:
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (c1:Column)-[:REFERENCES]->(c2:Column)
        MATCH (c1)<-[:HAS_COLUMN]-(t1:Table)
        MATCH (c2)<-[:HAS_COLUMN]-(t2:Table)
        RETURN t1.name AS from_table, c1.name AS from_col,
               t2.name AS to_table, c2.name AS to_col
        ORDER BY from_table
        LIMIT 5
    """)
    print("Join Path Examples:")
    for record in result:
        print(f"  {record['from_table']}.{record['from_col']} → {record['to_table']}.{record['to_col']}")

Join Path Examples:
  campaigns.owner_employee_id → employees.employee_id
  compensation.employee_id → employees.employee_id
  customer_addresses.customer_id → customers.customer_id
  customer_contacts.customer_id → customers.customer_id
  customers.account_owner_id → employees.employee_id


In [53]:
# Show vector indexes
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("SHOW VECTOR INDEXES")
    print("Vector Indexes:")
    for record in result:
        print(f"  - {record['name']}: label={record.get('labelsOrTypes', 'N/A')[0]}, state={record['state']}")

Vector Indexes:
  - column_vector_index: label=Column, state=ONLINE
  - schema_vector_index: label=Schema, state=ONLINE
  - table_vector_index: label=Table, state=ONLINE
